# 03 — Explainability: SHAP e controfattuali

Domande: quali fattori guidano il modello in generale? Perché ha rifiutato questo cliente? Cosa dovrebbe cambiare per ottenere il credito? Le spiegazioni sono stabili?

Prima di eseguire: `python -m credit_xai.pipeline` dalla cartella del progetto.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from credit_xai import config
from credit_xai.data import load_processed
from credit_xai.features import get_X_y
from credit_xai.model import CreditModel

sns.set_theme(style='whitegrid')
train, test = load_processed('train'), load_processed('test')
X_test, y_test = get_X_y(test)
model = CreditModel.load()

from credit_xai.explain.shap_explainer import ShapExplainer, stability
from credit_xai.explain.counterfactuals import CounterfactualGenerator
explainer = ShapExplainer(model)

## 1. Importanza globale

In [ ]:
imp = explainer.global_importance(test)
imp.index = [config.FEATURE_LABELS.get(f, f) for f in imp.index]
imp.head(12)[::-1].plot.barh(figsize=(7, 5), color='#4C72B0')
plt.xlabel('media |SHAP| (log-odds)'); plt.title('Cosa conta per il modello')

## 2. Direzione degli effetti

Ogni punto è un cliente: a destra il fattore aumenta il rischio.

In [ ]:
sv = explainer.shap_values(test)
top = imp.index[:8]
inv = {v: k for k, v in config.FEATURE_LABELS.items()}
long = sv[[inv.get(t, t) for t in top]].melt(var_name='feature', value_name='shap')
long['feature'] = long['feature'].map(config.FEATURE_LABELS)
plt.figure(figsize=(8, 5))
sns.stripplot(data=long, y='feature', x='shap', size=3, alpha=.5)
plt.axvline(0, c='grey'); plt.title('Distribuzione dei contributi SHAP')

## 3. Una decisione spiegata

In [ ]:
p = model.prob_default(test)
row = test.loc[[p.sort_values().index[-5]]]  # un cliente rifiutato
ex = explainer.explain_one(row)
print(f"Probabilità di default {ex['prob_default']:.0%} - approvato: {ex['approved']}")
contrib = pd.DataFrame(ex['contributions'])
contrib.head(8)[['label', 'value', 'shap', 'direction']]

In [ ]:
top = contrib.head(10)[::-1]
colors = ['#c0392b' if v > 0 else '#2471a3' for v in top['shap']]
plt.figure(figsize=(7, 5))
plt.barh(top['label'] + ' = ' + top['value'], top['shap'], color=colors)
plt.axvline(0, c='grey'); plt.xlabel('← riduce il rischio | aumenta il rischio →')

## 4. Controfattuali: cosa cambierebbe l'esito

In [ ]:
gen = CounterfactualGenerator(model, train)
for i, cf in enumerate(gen.generate(row), 1):
    changes = '; '.join(f"{c['label']}: {c['from']} → {c['to']}" for c in cf['changes'])
    print(f"{i}. {changes}  (rischio {cf['prob_default']:.0%})")

## 5. Stabilità delle spiegazioni

Perturbiamo di ±5% le variabili numeriche: le top-5 feature restano le stesse? (1 = sempre uguali)

In [ ]:
sample = test.sample(30, random_state=0)
scores = [stability(explainer, sample.iloc[[i]], n=20) for i in range(len(sample))]
print(f'Stabilità media: {sum(scores)/len(scores):.2f}')
pd.Series(scores).plot.hist(bins=10, title='Stabilità per cliente')

## Note

- Fattori più importanti: …
- Controfattuali sensati o no: …
- Stabilità: …